In [0]:
from pyspark.sql import functions as F

# =========================
# CONFIG
# =========================
STORAGE_ACCOUNT = "ragadziyada"
STORAGE_KEY = "qIXjwo7EGK8an84BPCk446tqY9L7K7r3a9W2WB+voe2vUCvW1SK3qc/7UGOicKyBAtrptYcVfTB7+AStvWZi0A=="

spark.conf.set(
    f"fs.azure.account.key.{STORAGE_ACCOUNT}.dfs.core.windows.net",
    STORAGE_KEY
)

ARTICLES_PATH = f"abfss://processed-p1@{STORAGE_ACCOUNT}.dfs.core.windows.net/articles_landing/"
CUSTOMERS_PATH = f"abfss://processed-p1@{STORAGE_ACCOUNT}.dfs.core.windows.net/customers_landing/"
TRANSACTIONS_PATH = f"abfss://processed-p1@{STORAGE_ACCOUNT}.dfs.core.windows.net/transactions_landing/"

# =========================
# LOAD DATA
# =========================
articles_df = spark.read.parquet(ARTICLES_PATH)
customers_df = spark.read.parquet(CUSTOMERS_PATH)
transactions_df = spark.read.parquet(TRANSACTIONS_PATH)

# =========================
# BASIC VALIDATION
# =========================
print("=== ROW COUNTS ===")
print("articles:", articles_df.count())
print("customers:", customers_df.count())
print("transactions:", transactions_df.count())

print("\n=== SCHEMAS ===")
print("ARTICLES")
articles_df.printSchema()

print("CUSTOMERS")
customers_df.printSchema()

print("TRANSACTIONS")
transactions_df.printSchema()

# =========================
# NULL REPORT
# =========================
def null_report(df, name):
    print(f"\n=== NULL REPORT: {name} ===")
    exprs = [F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]
    display(df.select(exprs))

null_report(articles_df, "articles")
null_report(customers_df, "customers")
null_report(transactions_df, "transactions")

# =========================
# DUPLICATE CHECKS
# =========================
print("\n=== DUPLICATE CHECKS ===")
print("Duplicate article_id rows:",
      articles_df.groupBy("article_id").count().filter("count > 1").count())

print("Duplicate customer_id rows:",
      customers_df.groupBy("customer_id").count().filter("count > 1").count())

txn_dup_cols = ["t_dat", "customer_id", "article_id", "price", "sales_channel_id"]
print("Duplicate transaction rows:",
      transactions_df.groupBy(txn_dup_cols).count().filter("count > 1").count())

# =========================
# SAMPLE DATA
# =========================
print("\n=== SAMPLE ARTICLES ===")
display(articles_df.limit(10))

print("\n=== SAMPLE CUSTOMERS ===")
display(customers_df.limit(10))

print("\n=== SAMPLE TRANSACTIONS ===")
display(transactions_df.limit(10))

=== ROW COUNTS ===
articles: 105542
customers: 1371980
transactions: 31788324

=== SCHEMAS ===
ARTICLES
root
 |-- article_id: string (nullable = true)
 |-- product_code: string (nullable = true)
 |-- prod_name: string (nullable = true)
 |-- product_type_no: string (nullable = true)
 |-- product_type_name: string (nullable = true)
 |-- product_group_name: string (nullable = true)
 |-- graphical_appearance_no: string (nullable = true)
 |-- graphical_appearance_name: string (nullable = true)
 |-- colour_group_code: string (nullable = true)
 |-- colour_group_name: string (nullable = true)
 |-- perceived_colour_value_id: string (nullable = true)
 |-- perceived_colour_value_name: string (nullable = true)
 |-- perceived_colour_master_id: string (nullable = true)
 |-- perceived_colour_master_name: string (nullable = true)
 |-- department_no: string (nullable = true)
 |-- department_name: string (nullable = true)
 |-- index_code: string (nullable = true)
 |-- index_name: string (nullable = true

article_id,product_code,prod_name,product_type_no,product_type_name,product_group_name,graphical_appearance_no,graphical_appearance_name,colour_group_code,colour_group_name,perceived_colour_value_id,perceived_colour_value_name,perceived_colour_master_id,perceived_colour_master_name,department_no,department_name,index_code,index_name,index_group_no,index_group_name,section_no,section_name,garment_group_no,garment_group_name,detail_desc
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,416



=== NULL REPORT: customers ===


customer_id,FN,Active,club_member_status,fashion_news_frequency,age,postal_code
0,895050,907576,6062,16009,15861,0



=== NULL REPORT: transactions ===


...

article_id,product_code,prod_name,product_type_no,product_type_name,product_group_name,graphical_appearance_no,graphical_appearance_name,colour_group_code,colour_group_name,perceived_colour_value_id,perceived_colour_value_name,perceived_colour_master_id,perceived_colour_master_name,department_no,department_name,index_code,index_name,index_group_no,index_group_name,section_no,section_name,garment_group_no,garment_group_name,detail_desc
0108775015,0108775,Strap top,253,Vest top,Garment Upper body,1010016,Solid,09,Black,4,Dark,5,Black,1676,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.
0108775044,0108775,Strap top,253,Vest top,Garment Upper body,1010016,Solid,10,White,3,Light,9,White,1676,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.
0108775051,0108775,Strap top (1),253,Vest top,Garment Upper body,1010017,Stripe,11,Off White,1,Dusty Light,9,White,1676,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.
0110065001,0110065,OP T-shirt (Idro),306,Bra,Underwear,1010016,Solid,09,Black,4,Dark,5,Black,1339,Clean Lingerie,B,Lingeries/Tights,1,Ladieswear,61,Womens Lingerie,1017,"Under-, Nightwear","Microfibre T-shirt bra with underwired, moulded, lightly padded cups that shape the bust and provide good support. Narrow adjustable shoulder straps and a narrow hook-and-eye fastening at the back. Without visible seams for greater comfort."
0110065002,0110065,OP T-shirt (Idro),306,Bra,Underwear,1010016,Solid,10,White,3,Light,9,White,1339,Clean Lingerie,B,Lingeries/Tights,1,Ladieswear,61,Womens Lingerie,1017,"Under-, Nightwear","Microfibre T-shirt bra with underwired, moulded, lightly padded cups that shape the bust and provide good support. Narrow adjustable shoulder straps and a narrow hook-and-eye fastening at the back. Without visible seams for greater comfort."
0110065011,0110065,OP T-shirt (Idro),306,Bra,Underwear,1010016,Solid,12,Light Beige,1,Dusty Light,11,Beige,1339,Clean Lingerie,B,Lingeries/Tights,1,Ladieswear,61,Womens Lingerie,1017,"Under-, Nightwear","Microfibre T-shirt bra with underwired, moulded, lightly padded cups that shape the bust and provide good support. Narrow adjustable shoulder straps and a narrow hook-and-eye fastening at the back. Without visible seams for greater comfort."
0111565001,0111565,20 den 1p Stockings,304,Underwear Tights,Socks & Tights,1010016,Solid,09,Black,4,Dark,5,Black,3608,Tights basic,B,Lingeries/Tights,1,Ladieswear,62,"Womens Nightwear, Socks & Tigh",1021,Socks and Tights,"Semi shiny nylon stockings with a wide, reinforced trim at the top. Use with a suspender belt. 20 denier."
0111565003,0111565,20 den 1p Stockings,302,Socks,Socks & Tights,1010016,Solid,13,Beige,2,Medium Dusty,11,Beige,3608,Tights basic,B,Lingeries/Tights,1,Ladieswear,62,"Womens Nightwear, Socks & Tigh",1021,Socks and Tights,"Semi shiny nylon stockings with a wide, reinforced trim at the top. Use with a suspender belt. 20 denier."
0111586001,0111586,Shape Up 30 den 1p Tights,273,Leggings/Tights,Garment Lower body,1010016,Solid,09,Black,4,Dark,5,Black,3608,Tights basic,B,Lingeries/Tights,1,Ladieswear,62,"Womens Nightwear, Socks & Tigh",1021,Socks and Tights,Tights with built-in support to lift the bottom. Black in 30 denier and light amber in 15 denier.
0111593001,0111593,Support 40 den 1p Tights,304,Underwear Tights,Socks & Tights,1010016,Solid,09,Black,4,Dark,5,Black,3608,Tights basic,B,Lingeries/Tights,1,Ladieswear,62,"Womens Nightwear, Socks & Tigh",1021,Socks and Tights,"Semi shiny tights that shape the tummy, thighs and calves while also encouraging blood circulation in the legs. Elasticated waist."



=== SAMPLE CUSTOMERS ===


customer_id,FN,Active,club_member_status,fashion_news_frequency,age,postal_code
00000dbacae5abe5e23885899a1fa44253a17956c6d1c3d25f88aa139fdfc657,null,null,ACTIVE,NONE,49,52043ee2162cf5aa7ee79974281641c6f11a68d276429a91f8ca0d4b6efa8100
0000423b00ade91418cceaf3b26c6af3dd342b51fd051eec9c12fb36984420fa,null,null,ACTIVE,NONE,25,2973abc54daa8a5f8ccfe9362140c63247c5eee03f1d93f4c830291c32bc3057
000058a12d5b43e67d225668fa1f8d618c13dc232df0cad8ffe7ad4a1091e318,null,null,ACTIVE,NONE,24,64f17e6a330a85798e4998f62d0930d14db8db1c054af6c9090f7dd3e38380dc
00005ca1c9ed5f5146b52ac8639a40ca9d57aeff4d1bd2c5feb1ca5dff07c43e,null,null,ACTIVE,NONE,54,5d36574f52495e81f019b680c843c443bd343d5ca5b1c222539af5973a23ae6d
00006413d8573cd20ed7128e53b7b13819fe5cfc2d801fe7fc0f26dd8d65a85a,1.0,1.0,ACTIVE,Regularly,52,25fa5ddee9aac01b35208d01736e57942317d756b32ddd4564743b005a805b1d
000064249685c11552da43ef22a5030f35a147f723d5b02ddd9fd22452b1f5a6,null,null,null,null,null,2c29ae653a9282cce4151bd87643c907644e09541abc28ae87dea0d1f6603b1c
0000757967448a6cb83efb3ea7a3fb9d418ac7adf2379d8cd0c725276a467a2a,null,null,ACTIVE,NONE,20,fe7b8e2b3fafb89ca90db17ffeeae0fd29b795d803f74961cf8f27adb16e67e4
00007d2de826758b65a93dd24ce629ed66842531df6699338c5570910a014cc2,1.0,1.0,ACTIVE,Regularly,32,8d6f45050876d059c830a0fe63f1a4c022de279bb68ce312615afb86814fec7a
00007e8d4e54114b5b2a9b51586325a8d0fa74ea23ef77334eaec4ffccd7ebcc,null,null,ACTIVE,NONE,20,2c29ae653a9282cce4151bd87643c907644e09541abc28ae87dea0d1f6603b1c
00008469a21b50b3d147c97135e25b4201a8c58997f78782a0cc706645e14493,null,null,ACTIVE,NONE,20,2c29ae653a9282cce4151bd87643c907644e09541abc28ae87dea0d1f6603b1c



=== SAMPLE TRANSACTIONS ===


t_dat,customer_id,article_id,price,sales_channel_id
2018-09-20,000058a12d5b43e67d225668fa1f8d618c13dc232df0cad8ffe7ad4a1091e318,0663713001,0.050830508474576264,2
2018-09-20,000058a12d5b43e67d225668fa1f8d618c13dc232df0cad8ffe7ad4a1091e318,0541518023,0.03049152542372881,2
2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699338c5570910a014cc2,0505221004,0.01523728813559322,2
2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699338c5570910a014cc2,0685687003,0.016932203389830508,2
2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699338c5570910a014cc2,0685687004,0.016932203389830508,2
2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699338c5570910a014cc2,0685687001,0.016932203389830508,2
2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699338c5570910a014cc2,0505221001,0.020322033898305086,2
2018-09-20,00083cda041544b2fbb0e0d2905ad17da7cf1007526fb4c73235dccbbc132280,0688873012,0.03049152542372881,1
2018-09-20,00083cda041544b2fbb0e0d2905ad17da7cf1007526fb4c73235dccbbc132280,0501323011,0.053372881355932204,1
2018-09-20,00083cda041544b2fbb0e0d2905ad17da7cf1007526fb4c73235dccbbc132280,0598859003,0.0457457627118644,2
